## Non-Spatial Data Visualization

This notebook builds the non-spatial data visualizations that we will include in our dashboard. We will continue to build on exploration conducted by Lam Thieu in Week 3. We will aim to visualize some of the trends between acres burned, median income, and Fair Plan enrollment

### Import data and install packages

In [21]:
import pandas as pd
import seaborn as sns
from pathlib import Path

# Set sns theme
sns.set_theme(
    style='white',
    font='sans-serif'
    )

In [22]:
root_fldr = Path.cwd().parent.parent
raw_fldr = root_fldr / 'data' / 'raw'
processed_fldr = root_fldr / 'data' / 'processed'

In [23]:
county_income = pd.read_csv(
    processed_fldr/'County_Income_CFP_2022_2024.csv',
    dtype={'GEOID': 'str'},
    )

county_fire = pd.read_csv(
    processed_fldr/'fire_by_county_2022_2024.csv',
    )
county_fire = county_fire.drop(columns=['Unnamed: 0', 'geometry'])

trends_data = pd.read_csv(
    processed_fldr/'County_Income_CFP_Fire_2022_2024.csv'
)

### Examine data and determine how to best visualize results

In [24]:
county_fire.head()

,fire_name,total_fire_acres,year,cause_description,fire_length_days,county_name,sqmi_burned,acres_burned,percent_fire_county
0,PARK,429603.0,2024,Arson,45,Butte County,82.650872,52896.557968,12.312893
1,PARK,429603.0,2024,Arson,45,Tehama County,588.606988,376708.472256,87.687580
2,YORK,93077.9,2023,Unknown/Unidentified,23,San Bernardino County,132.361888,84711.608635,91.011517
3,KELLY,89326.1,2023,Unknown/Unidentified,91,Del Norte County,120.007811,76804.999267,85.982707
4,MOSQUITO,76771.4,2022,Miscellaneous,46,El Dorado County,20.866813,13354.760600,17.395489


In [25]:
county_income.head()

,GEOID,county,year,Median_income,Income_MOE,policies,exposure
0,06001,Alameda,2022,122488,1231,2956,3047740652
1,06003,Alpine,2022,101125,17442,263,179049160
2,06005,Amador,2022,74853,6048,4041,2808434094
3,06007,Butte,2022,66085,2261,3722,2472656045
4,06009,Calaveras,2022,77526,3875,6983,4970383068


In [26]:
trends_data.head()

,county,policies_2022,policies_2023,policies_2024,policy_growth_pct,Median_income,acres_burned_total,income_group
0,Alameda,2956.0,3519.0,6774.0,129.161028,122488,2365.123992,Higher income
1,Alpine,263.0,297.0,346.0,31.558935,101125,32.593858,Higher income
2,Amador,4041.0,4742.0,5495.0,35.981193,74853,2828.727542,Middle income
3,Butte,3722.0,5878.0,8165.0,119.371306,66085,60036.571649,Lower income
4,Calaveras,6983.0,8520.0,9548.0,36.732064,77526,7390.358190,Middle income


#### Start by comparing acres burned and Fair Plan enrollment

In [27]:
# Join Fair Plan and fire data
fire_totals = (
    county_fire
    .groupby(['county_name', 'year'], as_index=False)
    .agg(acres_burned_total=('acres_burned', 'sum'),
         total_fires = ('fire_name', 'count'))
)

fire_totals.head()

,county_name,year,acres_burned_total,total_fires
0,Alameda County,2022,1095.641596,9
1,Alameda County,2024,1269.482396,4
2,Alpine County,2023,32.593858,1
3,Amador County,2022,2587.972049,2
4,Amador County,2023,168.395056,4


In [28]:

# Remove ' County' so names match county_table ('Butte County' -> 'Butte')
fire_totals['county'] = fire_totals['county_name'].str.replace(' County', '')

fire_totals = fire_totals[['county', 'year', 'acres_burned_total', 'total_fires']]


all_data = pd.merge(
    left=county_income, 
    right=fire_totals,
    on=['county', 'year']
)

all_data['acres_burned_total'] = all_data['acres_burned_total'].fillna(0)


In [29]:
all_data.head()

,GEOID,county,year,Median_income,Income_MOE,policies,exposure,acres_burned_total,total_fires
0,06001,Alameda,2022,122488,1231,2956,3047740652,1095.641596,9
1,06005,Amador,2022,74853,6048,4041,2808434094,2587.972049,2
2,06007,Butte,2022,66085,2261,3722,2472656045,819.379408,18
3,06009,Calaveras,2022,77526,3875,6983,4970383068,1894.984653,1
4,06013,Contra Costa,2022,120020,1496,1264,1386513747,449.502656,8


##### Build wide dataframe

In [32]:
data_wide = (
    all_data.
    pivot_table(
        index = 'county',
        columns= 'year',
        values=['policies', 'acres_burned_total', 'total_fires']
    )
)

data_wide.columns = [f"{year}_{val}" for val, year in data_wide.columns]

data_wide = data_wide.reset_index()

data_wide.head()

,county,2022_acres_burned_total,2023_acres_burned_total,2024_acres_burned_total,2022_policies,2023_policies,2024_policies,2022_total_fires,2023_total_fires,2024_total_fires
0,Alameda,1095.641596,NaN,1269.482396,2956.0,NaN,6774.0,9.0,NaN,4.0
1,Alpine,NaN,32.593858,NaN,NaN,297.0,NaN,NaN,1.0,NaN
2,Amador,2587.972049,168.395056,72.360437,4041.0,4742.0,5495.0,2.0,4.0,2.0
3,Butte,819.379408,180.361682,59036.830558,3722.0,5878.0,8165.0,18.0,6.0,16.0
4,Calaveras,1894.984653,139.482598,5355.890939,6983.0,8520.0,9548.0,1.0,2.0,2.0


We saw the largest increase in Fair Plan enrollment from 2023-2024. I'll start by comparing 2023 fires to 2024 enrollment

In [36]:
data_wide = data_wide.assign(
    policy_pct_change = lambda df: (df['2024_policies'] / df['2023_policies'] - 1) * 100
)

fp_fire_comp = data_wide[['county','policy_pct_change', '2023_acres_burned_total']]

fp_fire_comp.info()

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   county                   53 non-null     str    
 1   policy_pct_change        36 non-null     float64
 2   2023_acres_burned_total  41 non-null     float64
dtypes: float64(2), str(1)
memory usage: 1.4 KB
